# 픽셀 캐시 스모크 테스트 v3

v2 에서 드러난 문제를 고치고, 슬롯 설계가 맞는지 판정한다

| v2 문제 | v3 조치 |
|---|---|
| `glob(recursive=True)` 가 DICOM 80만개를 훑어 24분 소모 | `paths.find_input()` 깊이 제한 탐색 |
| 그래프가 노트북에 안 뜸 | 저장과 화면 출력을 둘 다 한다 |
| 가중 판별 실패(UNK) 5건 | TE 폴백과 STIR/TIRM 인식 추가 |
| 슬롯 커버리지 근거 부족 (30 스터디) | **전체 시리즈 헤더를 분류해 판정** |


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

t0 = time.time()
# knee-src 위치. paths 모듈 자체가 여기 있어서 find_input 을 쓸 수 없다.
# 알려진 후보만 확인한다. 재귀 glob 은 DICOM 80만개를 훑어 10분 넘게 걸린다.
for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src',
           '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c)
        print('src:', _c)
        break
else:
    raise FileNotFoundError('knee-src 를 못 찾음')

import paths

wheels = paths.find_input('knee-dicom-wheels')
print(f'wheels: {wheels}   ({time.time()-t0:.1f}s)')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                '--find-links', str(wheels),
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'],
               check=True)
import pydicom
print(f'pydicom {pydicom.__version__}   ({time.time()-t0:.1f}s)')

import config, dicom_io, cache
print(paths.describe())
config.verify_labels()
print(f'준비 완료 {time.time()-t0:.1f}s  (v2 는 여기까지 1454s 걸렸다)')

OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
report = {}


## 1. 전체 시리즈 분류

24,371개 시리즈의 헤더를 읽어 6슬롯 설계가 타당한지 본다
픽셀은 안 읽으므로 헤더만 훑는다


In [ ]:
import numpy as np, pandas as pd

tr_series = pd.read_csv(paths.TRAIN_SERIES_CSV)
train = pd.read_csv(paths.TRAIN_CSV)
print(f'{len(tr_series):,} series / {len(train):,} studies')

t = time.time()
rows = []
for r in tr_series.itertuples():
    sd = paths.TRAIN_SERIES_DIR / r.StudyInstanceUID / r.SeriesInstanceUID
    try:
        slot, meta = cache.classify_series(sd)
    except Exception as e:
        slot, meta = None, {'err': type(e).__name__}
    rows.append({'study': r.StudyInstanceUID, 'slot': slot or '(none)',
                 'plane': meta.get('plane'), 'w': meta.get('weighting'),
                 'fs': meta.get('fat_sat'), 'n_files': meta.get('n_files'),
                 'spacing': meta.get('spacing'),
                 'csv_plane': r.Anatomical_Plane, 'csv_fluid': r.Fluid_Sensitive})
cls = pd.DataFrame(rows)
print(f'{len(cls):,} series 분류 {time.time()-t:.0f}s')
cls.to_csv(OUT / 'series_classification_full.csv', index=False)


In [ ]:
print('=== 슬롯별 시리즈 수 ===')
print(cls.slot.value_counts().to_string())
print()
print('=== 가중 판별 결과 ===')
print(cls.w.value_counts(dropna=False).to_string())
unk = (cls.w == 'UNK').mean()
print(f'\nUNK 비율 {unk:.4f}   (v2 표본에서는 5/67 = 0.075 였다)')
report['unk_rate'] = round(float(unk), 4)
report['slot_counts'] = cls.slot.value_counts().to_dict()


In [ ]:
print('=== 배정 안 된 시리즈의 조합 (상위 15) ===')
none = cls[cls.slot == '(none)']
print(none.groupby(['plane','w','fs']).size().sort_values(ascending=False).head(15).to_string())
print(f'\n배정 안 됨 {len(none):,} / {len(cls):,}  ({len(none)/len(cls):.1%})')
report['unassigned_rate'] = round(len(none)/len(cls), 4)


## 2. 슬롯 커버리지 - 전체 스터디 기준

**이 표로 6슬롯 설계를 유지할지 결정한다**


In [ ]:
cov = (cls[cls.slot != '(none)']
       .groupby(['study','slot']).size().unstack(fill_value=0) > 0)
cov = cov.reindex(columns=list(dicom_io.SLOTS), fill_value=False)
cov = cov.reindex(train.StudyInstanceUID, fill_value=False)

summary = pd.DataFrame({'studies': cov.sum(), 'rate': cov.mean().round(3)})
print('=== 슬롯별 보유 스터디 비율 ===')
print(summary.to_string())
print()
print('=== 스터디당 채워진 슬롯 수 ===')
dist = cov.sum(axis=1).value_counts().sort_index()
print(dist.to_string())
print(f'\n평균 {cov.sum(axis=1).mean():.2f} / 6')
report['slot_rate'] = summary['rate'].to_dict()
report['n_filled_dist'] = {int(k): int(v) for k, v in dist.items()}


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].barh(range(len(summary))[::-1], summary['rate'], color='#4c72b0')
axes[0].set_yticks(range(len(summary))[::-1]); axes[0].set_yticklabels(summary.index)
axes[0].set_xlim(0, 1); axes[0].set_xlabel('fraction of studies')
axes[0].set_title('slot coverage', fontsize=10)
for i, v in enumerate(summary['rate']):
    axes[0].text(v + 0.01, len(summary)-1-i, f'{v:.0%}', va='center', fontsize=8)

axes[1].bar(dist.index, dist.values, color='#55a868')
axes[1].set_xlabel('slots filled per study'); axes[1].set_ylabel('studies')
axes[1].set_title('how many slots get filled', fontsize=10)
plt.tight_layout()
fig.savefig(OUT / 'slot_coverage.png', dpi=110, bbox_inches='tight')
plt.show()


## 3. 캐시 빌드 - 스터디 40개


In [ ]:
L = list(config.LABELS)
gold = train[train[L].notna().any(axis=1)].StudyInstanceUID.tolist()
rng = np.random.default_rng(0)
others = rng.choice(train.StudyInstanceUID.values, 30, replace=False).tolist()
targets = list(dict.fromkeys(gold[:10] + others + list(paths.CORRUPT_STUDIES)))

cfg = cache.CacheConfig()
arrs, infos = {}, {}
t = time.time()
for u in targets:
    a, info = cache.build_study(u, cfg)
    arrs[u], infos[u] = a, info
    if info.errors:
        print(f'  ! {u[-16:]}: {info.errors[0][:110]}')
el = time.time() - t
per = el / len(targets)
print(f'{len(targets)} studies  {el:.0f}s   스터디당 {per:.2f}s')
print(f'4,407개 단일 프로세스 추정 {per*len(train)/3600:.1f}시간')
report['timing'] = {'sec_per_study': round(per, 2),
                    'hours_single': round(per*len(train)/3600, 2),
                    'size': cache.estimate_size(len(train), cfg)}


In [ ]:
cr = pd.DataFrame([{'slot': s, 'applied': ok, 'mm': i.mm_per_px[s]}
                   for i in infos.values() for s, ok in i.crop_applied.items()])
print('=== 크롭 적용률 / 픽셀 피치 ===')
print(cr.groupby('slot').agg(applied=('applied','mean'),
                             mm_min=('mm','min'), mm_max=('mm','max')).round(4).to_string())
print(f'\n목표 {cfg.extent_mm}/{cfg.out_px} = {cfg.extent_mm/cfg.out_px:.4f}')
bad = cr[~cr.applied]
print(f'크롭 실패 {len(bad)} 건')
report['crop_fail'] = int(len(bad))


## 4. 이미지 확인

저장과 화면 출력을 둘 다 한다


In [ ]:
def grid(u, fname=None):
    a, info = arrs[u], infos[u]
    fig, axes = plt.subplots(len(cfg.slots), cfg.n_slices,
                             figsize=(cfg.n_slices*1.25, len(cfg.slots)*1.4))
    for i, slot in enumerate(cfg.slots):
        for j in range(cfg.n_slices):
            ax = axes[i, j]
            ax.imshow(a[i, j], cmap='gray', vmin=0, vmax=255)
            ax.set_xticks([]); ax.set_yticks([])
            if j == 0:
                filled = slot in info.slots_filled
                ax.set_ylabel(slot, fontsize=7, color='black' if filled else 'red')
    fig.suptitle(f'{u[-16:]}   {info.n_filled}/6 slots', fontsize=10)
    plt.tight_layout()
    if fname:
        fig.savefig(OUT / fname, dpi=95, bbox_inches='tight')
    plt.show()

for k, u in enumerate(list(infos)[:3]):
    grid(u, f'grid_{k}_{u[-10:]}.png')


In [ ]:
# 슬라이스 정렬 검증 - 기하 순서 vs 파일명 순서
# 같은 스케일로 비교하기 위해 둘 다 정규화 후 측정한다
u = list(infos)[0]
picked = cache.pick_series(paths.TRAIN_SERIES_DIR / u, cfg)
series = list(picked.values())[0]
files_named = sorted(series.glob('*.dcm'))
n = len(files_named)
sel_named = files_named[n//2-4 : n//2+5]

keyed = sorted(((dicom_io.slice_key(pydicom.dcmread(str(f), stop_before_pixels=True, force=True)), f)
                for f in files_named), key=lambda t: t[0])
sel_geo = [f for _, f in keyed[n//2-4 : n//2+5]]

def diffs(files):
    st = np.stack([pydicom.dcmread(str(f), force=True).pixel_array for f in files])
    st = dicom_io.normalise_stack(st).astype(int)
    return [np.abs(st[k]-st[k+1]).mean() for k in range(len(st)-1)]

dg, dn = diffs(sel_geo), diffs(sel_named)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(dg, marker='o', label=f'geometric  (mean {np.mean(dg):.1f})', color='#4c72b0')
ax.plot(dn, marker='s', label=f'filename   (mean {np.mean(dn):.1f})', color='crimson')
ax.set_xlabel('slice pair'); ax.set_ylabel('mean abs diff (normalised)')
ax.set_title('slice ordering - same series, same normalisation', fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT / 'slice_order.png', dpi=110, bbox_inches='tight')
plt.show()
print(f'기하 {np.mean(dg):.1f}  vs  파일명 {np.mean(dn):.1f}')
report['slice_order'] = {'geometric': round(float(np.mean(dg)), 2),
                         'filename': round(float(np.mean(dn)), 2)}


## 5. 판정


In [ ]:
(OUT / 'smoke_report_v3.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(json.dumps(report, indent=2, ensure_ascii=False))
print()
print('saved:', sorted(os.listdir(OUT)))
print(f'\n전체 실행 {time.time()-t0:.0f}s')
